# My STORM Paper Reproduction — Step 1: Data Understanding and Audit

I am using this notebook as the first reproducible step of my small study on event-aware modelling of electricity-load time series. I do **not** train a generative model yet.

## Goal

I will download the STORM training data, inspect its structure, load one station safely, and understand the meaning of the measured load, the bottom-up estimate, the missing-data flag, and the expert labels.

## What I should be able to explain after this notebook

- `S_original`: the published load measurement at a primary substation.
- `BU_original`: an independent bottom-up estimate of the same aggregate load, constructed from downstream information. It may have a different scale or offset from `S_original`.
- `missing`: a data-quality indicator supplied with the features when available.
- `label = 0`: normal operation; `label = 1`: an anomalous interval or a switching event; `label = 5`: uncertain.
- Why data are split by **station**, rather than randomly by timestamp: this tests whether a method generalises to unseen network locations.

**Scope of Step 1:** I focus on understanding and visualising the data. Cleaning, robust scaling of `BU_original`, and construction of the residual $\delta$ belong to Step 2.


## 1. Libraries and Settings

I use standard Python libraries, Pandas, and Plotly. Plotly keeps my time-series figure interactive, so I can zoom into suspected events.


In [ ]:
from pathlib import Path
from urllib.request import urlretrieve
from zipfile import ZipFile, is_zipfile

import pandas as pd
import plotly.graph_objects as go

pd.set_option("display.max_columns", 20)


## 2. Download the Training Dataset

For this data-understanding step, I download only the training split. I will introduce validation and test splits when I evaluate methods. The download cell checks whether the archive already exists and avoids downloading it again when possible.


In [ ]:
TRAIN_URL = "https://alldxpprdstor.blob.core.windows.net/large-content-media/Train.zip"
DATA_DIR = Path("storm_data")
ARCHIVE_PATH = DATA_DIR / "Train.zip"
TRAIN_DIR = DATA_DIR / "Train"

DATA_DIR.mkdir(exist_ok=True)

if not TRAIN_DIR.exists():
    if not ARCHIVE_PATH.exists() or not is_zipfile(ARCHIVE_PATH):
        print("Downloading the official STORM training dataset...")
        urlretrieve(TRAIN_URL, ARCHIVE_PATH)
    if not is_zipfile(ARCHIVE_PATH):
        raise ValueError("The downloaded file is not a valid ZIP archive. Delete it and run this cell again.")
    with ZipFile(ARCHIVE_PATH) as archive:
        archive.extractall(DATA_DIR)
    print("Download and extraction completed.")
else:
    print("Training data already exists; download skipped.")

## 3. Inspect the Folder Structure

- The `X` folder contains timestamps, the substation load measurement (`S_original`), the bottom-up estimate (`BU_original`), and optional data-quality information.
- The `y` folder contains expert-provided labels.
- Each filename is an anonymised station identifier; for example, `100.csv`.

`S_original` and `BU_original` describe the same broad load, but they are produced differently. I will not compare them as a raw difference until I align their scale and offset in Step 2.


In [ ]:
X_DIR = TRAIN_DIR / "X"
Y_DIR = TRAIN_DIR / "y"

x_files = sorted(X_DIR.glob("*.csv"), key=lambda path: int(path.stem))
y_files = sorted(Y_DIR.glob("*.csv"), key=lambda path: int(path.stem))
x_ids = {path.stem for path in x_files}
y_ids = {path.stem for path in y_files}

print(f"Number of X files: {len(x_files)}")
print(f"Number of y files: {len(y_files)}")
print(f"X and y station IDs match: {x_ids == y_ids}")
print("First station IDs:", [path.stem for path in x_files[:10]])

## 4. Load One Station Safely

The extra `Unnamed` columns are CSV-saved index columns rather than physical features, so I remove them. Before merging `X` and `y`, I verify that the two files contain exactly the same timestamps. This prevents an `inner` merge from silently dropping data.


In [ ]:
def remove_saved_index_columns(frame):
    return frame.loc[:, ~frame.columns.str.startswith("Unnamed")].copy()


def parse_timestamp(series):
    return pd.to_datetime(series, format="mixed", dayfirst=True, utc=True)


def load_station(station_id, split_dir=TRAIN_DIR):
    station_id = str(station_id)
    x_path = split_dir / "X" / f"{station_id}.csv"
    y_path = split_dir / "y" / f"{station_id}.csv"

    if not x_path.exists() or not y_path.exists():
        raise FileNotFoundError(f"X or y file is missing for station {station_id}.")

    features = remove_saved_index_columns(pd.read_csv(x_path))
    labels = remove_saved_index_columns(pd.read_csv(y_path))

    required_feature_columns = {"M_TIMESTAMP", "S_original", "BU_original"}
    missing_columns = required_feature_columns.difference(features.columns)
    if missing_columns:
        raise ValueError(f"Station {station_id} is missing feature columns: {sorted(missing_columns)}")
    if not {"M_TIMESTAMP", "label"}.issubset(labels.columns):
        raise ValueError(f"Station {station_id} has an invalid label file.")

    features["M_TIMESTAMP"] = parse_timestamp(features["M_TIMESTAMP"])
    labels["M_TIMESTAMP"] = parse_timestamp(labels["M_TIMESTAMP"])

    if features["M_TIMESTAMP"].duplicated().any() or labels["M_TIMESTAMP"].duplicated().any():
        raise ValueError(f"Station {station_id} contains duplicate timestamps.")
    feature_timestamps = features["M_TIMESTAMP"].sort_values(ignore_index=True)
    label_timestamps = labels["M_TIMESTAMP"].sort_values(ignore_index=True)
    if not feature_timestamps.equals(label_timestamps):
        raise ValueError(f"X and y timestamps do not match for station {station_id}.")

    if "missing" not in features.columns:
        features["missing"] = 0  # This column is optional in some files.

    station = features.merge(labels, on="M_TIMESTAMP", how="inner", validate="one_to_one")
    return station.sort_values("M_TIMESTAMP").reset_index(drop=True)


SAMPLE_STATION = x_files[0].stem  # Change this ID to inspect another station.
station = load_station(SAMPLE_STATION)
station.head()


## 5. Initial Data Audit

In this section, we examine the number of rows, time range, sampling interval, missing values, and label distribution.


In [ ]:
sampling_interval = station["M_TIMESTAMP"].diff().dropna().mode().iloc[0]

summary = pd.Series({
    "station_id": SAMPLE_STATION,
    "rows": len(station),
    "start": station["M_TIMESTAMP"].min(),
    "end": station["M_TIMESTAMP"].max(),
    "typical_sampling_interval": sampling_interval,
    "missing_S": station["S_original"].isna().sum(),
    "missing_BU": station["BU_original"].isna().sum(),
    "rows_flagged_missing": int(station["missing"].fillna(0).sum()),
})

display(summary.to_frame("value"))
display(station.dtypes.to_frame("dtype"))
display(station["label"].value_counts(dropna=False).sort_index().rename("count").to_frame())

## 6. Find a Suitable Station for Visualization

To make sure the educational plot is not empty of events, we select the station with the largest number of points labeled `1` in the training dataset. This selection is only for visualization and is not part of model training.


In [ ]:
def count_labels(label_path):
    labels = remove_saved_index_columns(pd.read_csv(label_path, usecols=lambda name: not name.startswith("Unnamed")))
    counts = labels["label"].value_counts()
    return {
        "station_id": label_path.stem,
        "normal_0": int(counts.get(0, 0)),
        "event_1": int(counts.get(1, 0)),
        "uncertain_5": int(counts.get(5, 0)),
    }

label_summary = pd.DataFrame(count_labels(path) for path in y_files)
label_summary = label_summary.sort_values("event_1", ascending=False).reset_index(drop=True)
display(label_summary.head(10))

PLOT_STATION = label_summary.loc[0, "station_id"]
plot_data = load_station(PLOT_STATION)
print("Selected station for visualization:", PLOT_STATION)

## 7. Interactive Overview: Load, Bottom-up Estimate, and Labels

I use this figure to inspect the two unscaled signals. Their visual difference is expected because `BU_original` can have a different scale and offset. Red points indicate an expert-labelled anomalous interval or switching event; orange points indicate uncertain labels. I use the zoom and hover tools before drawing conclusions about individual events.


In [ ]:
events = plot_data.loc[plot_data["label"] == 1]
uncertain = plot_data.loc[plot_data["label"] == 5]

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=plot_data["M_TIMESTAMP"],
    y=plot_data["S_original"],
    mode="lines",
    name="Measured load (S_original)",
    line=dict(width=1, color="#1f77b4"),
))

fig.add_trace(go.Scatter(
    x=plot_data["M_TIMESTAMP"],
    y=plot_data["BU_original"],
    mode="lines",
    name="Bottom-up estimate (BU_original, unscaled)",
    line=dict(width=1, color="#7f7f7f"),
    opacity=0.8,
))

fig.add_trace(go.Scatter(
    x=events["M_TIMESTAMP"],
    y=events["S_original"],
    mode="markers",
    name="Label 1: event or anomaly",
    marker=dict(color="crimson", size=5),
))

fig.add_trace(go.Scatter(
    x=uncertain["M_TIMESTAMP"],
    y=uncertain["S_original"],
    mode="markers",
    name="Label 5: uncertain",
    marker=dict(color="darkorange", size=5),
))

fig.update_layout(
    title=f"STORM training station {PLOT_STATION}: raw signals and labels",
    xaxis_title="Timestamp (UTC)",
    yaxis_title="Load value (published units)",
    template="plotly_white",
    width=1200,
    height=520,
    hovermode="x unified",
    legend=dict(x=0.01, y=0.99),
)
fig.update_xaxes(rangeslider_visible=True)
fig.show()


## 8. My Observations Before Step 2

1. Do `S_original` and `BU_original` generally follow a similar temporal trend, despite their possible scale difference?
2. Do red labels appear mostly as isolated points or as contiguous time intervals?
3. Why might a switching event be a real operational event rather than simply bad data?
4. Why would the raw difference `S_original - BU_original` be misleading before scale and offset correction?

**My next step:** reproduce the paper's preprocessing procedure: remove invalid observations, robustly align `BU_original` with `S_original`, apply sign correction where necessary, and construct the residual vector $\delta$.
